# Research 30 — Cross-seed VIP × region probability average

**Hypothesis:** the VIP × region uplift is similar across two CV seeds; averaging their aligned OOF probability matrices may smooth split-specific variance. Compare fixed 50/50 model averaging at a fixed 20% contribution to the base with each single-seed blend. Both components are OOF per row. This is a training-OOF synthesis, not independent validation; no fitting, weight search, or Test access.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold

ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "data/train.csv").is_file()
)
y = pd.read_csv(ROOT / "data/train.csv", usecols=["segment"]).segment.to_numpy(dtype=int)


def load_probs(filename):
    frame = pd.read_csv(ROOT / "predictions" / filename)
    assert len(frame) == len(y) and np.array_equal(frame.segment.to_numpy(), y), filename
    values = frame[[f"prob_{i}" for i in range(4)]].to_numpy(dtype=float)
    assert np.isfinite(values).all() and (values >= 0).all(), filename
    return values / values.sum(axis=1, keepdims=True)


p_base = load_probs("segment-oof.csv")
p_seed42 = load_probs("segment-vip-region-balanced-oof.csv")
p_seed2025 = load_probs("segment-vip-region-seed-2025-oof.csv")
p_average = 0.5 * p_seed42 + 0.5 * p_seed2025
candidates = {
    "Current reference": p_base,
    "80/20 VIP × region seed 42": 0.8 * p_base + 0.2 * p_seed42,
    "80/20 VIP × region seed 2025": 0.8 * p_base + 0.2 * p_seed2025,
    "80/20 VIP × region cross-seed average": 0.8 * p_base + 0.2 * p_average,
}
fold_sets = {
    "seed42": list(
        StratifiedKFold(n_splits=3, shuffle=True, random_state=42).split(np.zeros(len(y)), y)
    ),
    "seed2025": list(
        StratifiedKFold(n_splits=3, shuffle=True, random_state=2025).split(np.zeros(len(y)), y)
    ),
}
rows = []
for name, probs in candidates.items():
    pred = probs.argmax(axis=1)
    row = {
        "candidate": name,
        "macro_f1": f1_score(y, pred, average="macro"),
        "weighted_f1": f1_score(y, pred, average="weighted"),
    }
    for seed_name, folds in fold_sets.items():
        for fold, (_, idx) in enumerate(folds, start=1):
            row[f"{seed_name}_fold_{fold}_macro_f1"] = f1_score(y[idx], pred[idx], average="macro")
    rows.append(row)
results = pd.DataFrame(rows)
display(results.round(6))
results.to_csv(ROOT / "assets/segment-vip-region-crossseed.csv", index=False)
pd.DataFrame(
    candidates["80/20 VIP × region cross-seed average"], columns=[f"prob_{i}" for i in range(4)]
).assign(segment=y).to_csv(
    ROOT / "predictions/segment-vip-region-crossseed-blend-oof.csv", index=False
)
print("Fixed cross-seed OOF blend complete; Test not loaded.")

,candidate,macro_f1,weighted_f1,seed42_fold_1_macro_f1,seed42_fold_2_macro_f1,seed42_fold_3_macro_f1,seed2025_fold_1_macro_f1,seed2025_fold_2_macro_f1,seed2025_fold_3_macro_f1
0,Current reference,0.750895,0.764683,0.749445,0.749821,0.753390,0.750797,0.749412,0.752462
1,80/20 VIP × region seed 42,0.751552,0.765087,0.750030,0.750601,0.753998,0.750568,0.750774,0.753298
2,80/20 VIP × region seed 2025,0.751547,0.765136,0.750114,0.750342,0.754156,0.751659,0.750233,0.752741
3,80/20 VIP × region cross-seed average,0.751706,0.765361,0.749909,0.750991,0.754192,0.751228,0.750859,0.753021


Fixed cross-seed OOF blend complete; Test not loaded.
